# Libraries

In [1]:
import pickle
import pandas as pd
import numpy as np
import unicodedata
import re
pd.set_option('display.max_columns', None)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from pysr import PySRRegressor
import matplotlib.pyplot as plt
import itertools
import os

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


# Read dataset

In [2]:
filename = '../../dataset/dataset_2_rv3.xlsx'
dataset = pd.read_excel(filename)
def clean_column_name(name):
    name = unicodedata.normalize('NFKD', str(name)).encode('ASCII', 'ignore').decode('ASCII')
    name = name.lower()
    name = name.replace(' ', '_')
    name = re.sub(r'[^\w_]', '', name)
    name = re.sub(r'__+', '_', name)
    name = name.strip('_')
    return name
dataset.columns = [clean_column_name(col) for col in dataset.columns]

In [3]:
x = dataset[[
                'diametro_da_armadura_mm', 'resistencia_do_aco_fy_mpa',
                'resistencia_a_tracao_do_concreto_mpa', 'cobrimento_mm',
                'taxa_de_armadura_conforme_mc2020',
                'tensao_na_armadura_menos_tension_stiffening_variavel_mpa',
                'fib_model_code_2010', 'eurocode_2',
                'nbr_6118_wk1_fissuracao_estabilizada',
                'nbr_6118_wk2_fissuracao_nao_estabilizada', 'clark_1956',
                'desayiganesan_1985', 'oh_kang_1987', 'caldas_1997',
                'gergely_e_lutz_1968', 'equacao_15_do_artigo_de_belarbi_e_hsu_1994',
                'beta_conforme_fields_e_bischoff_2004', 'slip_de_yuan_et_al_2025',
                'slip_de_biscaia_e_soares_2020'
            ]]  
x.head()

,diametro_da_armadura_mm,resistencia_do_aco_fy_mpa,resistencia_a_tracao_do_concreto_mpa,cobrimento_mm,taxa_de_armadura_conforme_mc2020,tensao_na_armadura_menos_tension_stiffening_variavel_mpa,fib_model_code_2010,eurocode_2,nbr_6118_wk1_fissuracao_estabilizada,nbr_6118_wk2_fissuracao_nao_estabilizada,clark_1956,desayiganesan_1985,oh_kang_1987,caldas_1997,gergely_e_lutz_1968,equacao_15_do_artigo_de_belarbi_e_hsu_1994,beta_conforme_fields_e_bischoff_2004,slip_de_yuan_et_al_2025,slip_de_biscaia_e_soares_2020
0,20.0,500,4.120341,25.0,0.0213,120.527746,0.125734,0.125292,0.262468,0.157380,0.181006,0.312684,0.246013,0.129998,0.241528,0.149956,0.116566,0.109686,0.082007
1,20.0,500,4.120341,25.0,0.0213,223.847983,0.176982,0.176359,0.344489,0.271111,0.244363,0.410398,0.338924,0.174746,0.317005,0.204348,0.197288,0.193326,0.156266
2,20.0,500,4.120341,25.0,0.0213,242.052198,0.184078,0.183430,0.355846,0.289281,0.253135,0.423928,0.351788,0.180941,0.327456,0.212052,0.208041,0.205301,0.167096
3,20.0,500,4.120341,25.0,0.0213,325.147190,0.218769,0.217999,0.411368,0.386596,0.296022,0.490072,0.414682,0.211232,0.378549,0.250139,0.259368,0.265222,0.221958
4,20.0,500,4.120341,25.0,0.0213,447.784457,0.286574,0.285566,0.519888,0.617471,0.379848,0.619355,0.537611,0.270437,0.478411,0.311870,0.354810,0.388948,0.338382


In [4]:
y = dataset[['wexp_menos_wfib2020ts']]
y.head()

,wexp_menos_wfib2020ts
0,0.057870
1,0.039217
2,0.036942
3,0.006931
4,-0.030363


# Symbolic Regression

### Grid Search parameters

In [5]:
size_test = 0.2                             # Proportion of the dataset to include in the test split

In [6]:
n_folds = 10                                # Number of folds for cross-validation
parsimony_levels = [0.001, 0.01]            # Regularization levels to control model complexity
max_size_levels = [20, 30]                  # Maximum size of the equations
pop_size_levels = [250, 500]                # Population size for the evolutionary algorithm
ncycles_per_iteration_levels = [1000]       # Number of cycles per iteration
niterations_levels = [500]                  # Number of iterations
weight_optimize_levels = [0.001]            # Weight optimization levels

In [7]:
combs_ = list(itertools.product(parsimony_levels, max_size_levels, pop_size_levels, ncycles_per_iteration_levels, niterations_levels, weight_optimize_levels))
da = pd.DataFrame(combs_, columns=['parsimony', 'max_size', 'pop_size', 'ncycles_per_iteration', 'niterations', 'weight_optimize'])
da

,parsimony,max_size,pop_size,ncycles_per_iteration,niterations,weight_optimize
0,0.001,20,250,1000,500,0.001
1,0.001,20,500,1000,500,0.001
2,0.001,30,250,1000,500,0.001
3,0.001,30,500,1000,500,0.001
4,0.010,20,250,1000,500,0.001
5,0.010,20,500,1000,500,0.001
6,0.010,30,250,1000,500,0.001
7,0.010,30,500,1000,500,0.001


### Train and Test

In [8]:
dx = []
for j in range(n_folds):
   # Split the data in training and testing sets
   x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=size_test)
   # Standardize the features using StandardScaler z-score normalization
   scaler = StandardScaler()
   scaler.fit(x_train)
   x_train_scaled = scaler.transform(x_train)
   x_train_scaled = pd.DataFrame(x_train_scaled, columns=x.columns)
   x_test_scaled = scaler.transform(x_test)
   x_test_scaled = pd.DataFrame(x_test_scaled, columns=x.columns)
   
   for i, row in da.iterrows():
      # Save the scaler parameters
      dict_res = {}
      for i, col_name in enumerate(x_train.columns):
         dict_res[col_name] = {"avg": scaler.mean_[i], "std_dev": scaler.scale_[i]}
      
      # Train the PySR model
      model = PySRRegressor(
                              parsimony=row['parsimony'],           
                              maxsize=int(row['max_size']),
                              maxdepth=30,
                              population_size=int(row['pop_size']),
                              ncycles_per_iteration=int(row['ncycles_per_iteration']),
                              niterations=int(row['niterations']), 
                              adaptive_parsimony_scaling=50,
                              deterministic=False, 
                              should_optimize_constants=True,
                              optimizer_algorithm="BFGS",
                              optimizer_nrestarts=3,
                              binary_operators=["+", "-", "*", "/"],
                              unary_operators=["sin", "cos", "exp", "sqrt", "log", "abs", "square", "cube"],
                              weight_optimize=row['weight_optimize'],
                              verbosity=0,
                              progress=False,
                              parallelism='multithreading',
                              early_stop_condition=1e-8,
                           )
      model.fit(x_train_scaled, y_train)
      
      # Statistics and results
      y_pred_train = model.predict(x_train_scaled)
      y_pred_test = model.predict(x_test_scaled)
      rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
      rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
      r2_train = r2_score(y_train, y_pred_train)
      r2_test = r2_score(y_test, y_pred_test)
      print(f"Train RMSE: {rmse_train}")
      print(f"Test RMSE: {rmse_test}")
      print(f"Train R2: {r2_train}")
      print(f"Test R2: {r2_test}")
      best_eq = model.get_best()
      best_equation = best_eq['equation']
      print(f"Best Equation: {best_equation}")
      print("\n\n")
      dict_res['model'] = {
                              "train_rmse": rmse_train,
                              "test_rmse": rmse_test,
                              "train_r2": r2_train,
                              "test_r2": r2_test,
                              "best_equation": best_equation,
                              "scaler_params": scaler,
                              "model_object": model
                          }
      dx.append(dict_res)

Train RMSE: 0.053994735793867335
Test RMSE: 0.0637451442859482
Train R2: 0.6475155718323629
Test R2: 0.391875605323431
Best Equation: (((tensao_na_armadura_menos_tension_stiffening_variavel_mpa + (eurocode_2 / 0.57883465)) - slip_de_biscaia_e_soares_2020) - caldas_1997) * -0.056128457



Train RMSE: 0.05399473565577435
Test RMSE: 0.06374538837733769
Train R2: 0.6475155736353397
Test R2: 0.39187094808359446
Best Equation: (eurocode_2 - (((caldas_1997 + slip_de_biscaia_e_soares_2020) - tensao_na_armadura_menos_tension_stiffening_variavel_mpa) * 0.5788262)) * -0.09696948



Train RMSE: 0.0539947357375041
Test RMSE: 0.0637466993969169
Train R2: 0.6475155725682555
Test R2: 0.39184593365313714
Best Equation: ((tensao_na_armadura_menos_tension_stiffening_variavel_mpa + ((eurocode_2 / 0.57878894) - slip_de_biscaia_e_soares_2020)) - caldas_1997) * -0.05612994



Train RMSE: 0.05399473507667842
Test RMSE: 0.06374593409258014
Train R2: 0.6475155811961617
Test R2: 0.39186053582564206
Best Equation

### Save data

In [9]:
base_filename = os.path.basename(filename).replace('.xlsx', '')  # 'dataset_1_rv3'
aux = [{'metadados': 'index [1] data of model trained fold=0 grid=0, index [2] data of model trained fold=0 grid=1, ...'}]
results = aux + dx
with open(f'results_pysr_model_8020_fib2020ts_{base_filename}.pkl', 'wb') as f:
    pickle.dump(results, f)